# P4-M1.1 T4 correctness and timing smoke
Use the existing Kaggle Notebook. Select Tesla T4, enable Internet and GITHUB_TOKEN, then Save & Run All. The smoke uses one mapping and eight exposures per configuration. Attach completed saved Output to retry publication.

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import sys
import tempfile

remote = 'https://github.com/ArcheLabs/mini-cells.git'
repo = Path('/kaggle/working/minicells-p4-m11')
branch = 'research/p4-m11-m2-preparation'
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth=1', '--branch', branch, remote, str(repo)],
                   check=True, timeout=300)
source = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip()
base = repo / 'research/projects/cellular-language-models/experiments/p4-peer-titans-collaboration'
out = Path('/kaggle/working/p4-m1-' + source[:12])
manifest_path = out / 'run-manifest.json'
if not manifest_path.exists():
    for candidate in Path('/kaggle/input').rglob(out.name + '/run-manifest.json'):
        previous = json.loads(candidate.read_text())
        if previous.get('status') == 'PASS':
            shutil.copytree(candidate.parent, out)
            break
if not manifest_path.exists():
    subprocess.run([sys.executable, str(base/'m1.1-capacity/gpu_smoke.py'), str(out)],
                   cwd=repo, check=True, timeout=600)
manifest = json.loads(manifest_path.read_text())
if manifest['status'] != 'PASS':
    raise RuntimeError('P4-M1.1 T4 gate failed. Inspect retained run-manifest.json.')
runner = base/'m1.1-capacity'
for key, name in [('runner_sha256', 'gpu_smoke.py'), ('smoke_sha256', 'smoke.py')]:
    if manifest[key] != hashlib.sha256((runner/name).read_bytes()).hexdigest():
        raise RuntimeError('P4-M1.1 output source mismatch.')
if manifest['source_commit'] != source or 'T4' not in manifest['gpu']:
    raise RuntimeError('P4-M1.1 output execution identity mismatch.')
sys.path.insert(0, str(runner))
from gpu_smoke import report
report_text = report(manifest)
(out/'REPORT.md').write_text(report_text)
sys.path.insert(0, str(repo))
from scripts.research.publish_experiment_results import ExperimentSpec, prepare_artifacts, push_results

run_id = out.name
result_branch = 'kaggle/' + run_id
with tempfile.TemporaryDirectory(prefix='p4-publish-') as tmp:
    root = Path(tmp)/'repo'
    subprocess.run(['git', 'clone', '--quiet', '--shared', str(repo), str(root)], check=True, timeout=180)
    subprocess.run(['git', 'remote', 'set-url', 'origin', remote], cwd=root, check=True)
    stage = root/'results/p4-m11-t4'
    stage.mkdir(parents=True)
    (stage/'run-manifest.json').write_text(json.dumps(manifest, indent=2))
    (stage/'REPORT.md').write_text(report_text)
    (stage/'decision.json').write_text(json.dumps({
        'format': 'minicells.p4-m11-gpu-smoke.v1', 'status': manifest['status'],
        'source_commit': source, 'run_id': run_id, 'device': manifest['gpu'],
    }))
    spec = ExperimentSpec('results/p4-m11-t4', 'artifacts/experiments/p4-peer-titans/'+run_id,
                          ('decision.json', 'run-manifest.json', 'REPORT.md'), result_branch, 'minicells.p4-m11-gpu-smoke.v1')
    folder = prepare_artifacts(root, 'p4-peer-titans', spec, None)
    push_results(root, folder, 'p4-peer-titans', result_branch, 'GITHUB_TOKEN', 'origin/main')
    sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=root, text=True).strip()
(out/'publication.json').write_text(json.dumps({'status':'REMOTE_SHA_VERIFIED','commit':sha,'branch':result_branch}))
